# 2026.11.01
# 프로그래머스 복습

- 작성자 : 최현준
- 내용 : 코딩테스트

In [28]:
#간단한 식 계산하기

def solution(binomial):

    if "+" in binomial:
        a = binomial.split("+")
        return int(a[0]) + int(a[1])
    elif "-" in binomial:
        a = binomial.split("-")
        return int(a[0]) - int(a[1])
    elif "*" in binomial:
        a = binomial.split("*")
        return int(a[0]) * int(a[1])


# def solution(binomial):
#     a, op, b = binomial.split()
#     a, b = int(a), int(b)
#
#     if op == '+':
#         return a + b
#     elif op == '-':
#         return a - b
#     else:
#         return a * b


solution("0 - 7777")

-7777

In [63]:
#문자열 잘라서 정렬하기

def solution(myString):
    s = []
    a = sorted(myString.split('x'))
    for i in a:
        if i == '':
            continue
        else:
            s.append(i)
    return s



solution("axbxcxdx")

['a', 'b', 'c', 'd']

In [65]:
# x사이의 개수

def solution(myString):
    answer = []
    a = myString.split('x')
    for i in a:
        answer.append(len(i))
    return answer

solution("oxooxoxxox")

[1, 2, 1, 0, 1, 0]

In [67]:
#공백으로 구분하기1

def solution(my_string):
    return my_string.split(' ')

solution("i love you")

['i', 'love', 'you']

In [70]:
#ad제거하기

def solution(strArr):
    answer = []
    for i in strArr:
        if 'ad' in i:
            continue
        else:
            answer.append(i)
    return answer

solution(["there","are","no","a","ds"])

['there', 'are', 'no', 'a', 'ds']

In [79]:
#n보다 커질 때까지 더하기

def solution(numbers, n):
    answer = 0
    for i in numbers:
        if answer <= n:
            answer += i

    return answer




solution([34, 5, 71, 29, 100, 34], 123)

139

# 인공지능 복습

In [84]:
import random, math

def make(n, rng):
    # 키(cm)와 무게(kg)로 고양이/개 구별 — 겹치는 구간이 있다
    data = []
    for _ in range(n):
        cat = rng.random() < 0.5 # 50% 확률로 고양이
        h = rng.gauss(25 if cat else 45, 8) #키 : 고양이 평균 25cm, 개 평균 45cm, 흔들림 8(흔들림 표준편차? : 값이 평균에서 얼마나 퍼져있는지를 말함)
        w = rng.gauss(4 if cat else 15, 4) # 무게: 고양이 평균 4kg, 개 평균 15kg, 흔들림 4
        data.append(([h, w], 1 if cat else 0)) # ([키, 무게], 정답) — 고양이 1, 개 0
    return data


def train(data, rounds=200): #확률적 경사 하강법(SGD)
    w, b = [0.0, 0.0], 0.0 # 키 가중치, 무게 가중치, 절편을 0에서 시작
    for _ in range(rounds): # 데이터 전체를 200바퀴 (에폭 200)
        for x, y in data: # 한 마리씩 ([45.217921398627645, 13.203938333584667], 0),
                                  # ([25.0744786117629, 2.014721552139072], 1) <-데이터
                                  # x값에는 [키,몸무게] , y값에는 0,1 고양이인지 개인지
            z = sum(wi*xi for wi, xi in zip(w, x)) / 20 + b #로짓? 점수? 아직 확률이 아닌 날것의 점수
            p = 1 / (1 + math.exp(-max(-60, min(60, z)))) #예측확률
            g = p - y #오차(그래디언트 = 기울기)
            w = [wi - 0.05*g*xi for wi, xi in zip(w, x)]
            b -= 0.05 * g
    return w, b

def test(model, data):
    w, b = model
    ok = 0
    for x, y in data:
        z = sum(wi*xi for wi, xi in zip(w, x)) / 20 + b
        ok += (1 if z > 0 else 0) == y
    return ok / len(data) #맞춘 마리 수

big = make(2000, random.Random(999))    # 실전에서 만날 데이터(2000)
print("학습 데이터  정확도(10번 평균)   가장 낮았을 때 ~ 가장 높았을 때")
for n in (5, 10, 50, 200, 1000):
    accs = [test(train(make(n, random.Random(s))), big) for s in range(10)]
    print(f"  {n:4d}장       {sum(accs)/len(accs):.1%}"
          f"            {min(accs):.1%} ~ {max(accs):.1%}")

학습 데이터  정확도(10번 평균)   가장 낮았을 때 ~ 가장 높았을 때
     5장       73.9%            40.5% ~ 94.0%
    10장       88.5%            68.0% ~ 96.8%
    50장       95.9%            93.7% ~ 97.3%
   200장       96.0%            91.5% ~ 97.5%
  1000장       96.5%            95.2% ~ 97.4%


# 2026.11.03
인공지능/딥러닝 복습

- 작성자 : 최현준
- 내용 : [코딩테스트](file:///Users/hyjun/Downloads/textbook-ml-dl-previous-98a6e06/deeplearning/ch08_features.html)

In [6]:
# 데이터셋 = 특징 벡터들의 표 (행=대상 하나, 열=특징 하나)
#            [면적, 방수, 역거리, 연식]   가격(라벨 y)
houses = [
    ([84, 3, 1.2, 15], 6.2),
    ([59, 2, 0.4, 3],  7.1),
    ([115, 4, 2.5, 22], 5.8),
]

# 두 집이 얼마나 '비슷한가' = 벡터 사이 거리 (공간 속 점!)
def distance(a, b):
    return sum((p - q) ** 2 for p, q in zip(a, b)) ** 0.5


x0, x1, x2 = houses[0][0], houses[1][0], houses[2][0]
print(x0,x1,x2)
print(f"집0-집1 거리: {distance(x0, x1):.1f}")
print(f"집0-집2 거리: {distance(x0, x2):.1f}")
# 거리가 가까울수록 비슷한 집 — '비슷함'이 숫자가 됐다!

[84, 3, 1.2, 15] [59, 2, 0.4, 3] [115, 4, 2.5, 22]
집0-집1 거리: 27.8
집0-집2 거리: 31.8


In [7]:
# 훈련/테스트 점수로 모델 상태 진단하기
def diagnose(train_acc, test_acc):
    gap = train_acc - test_acc
    if train_acc < 0.7:
        return "과소적합 — 훈련조차 못 맞힘 (더 배워야)"
    if gap > 0.15:
        return "과적합 — 외웠음! (격차가 큼)"
    return "양호 — 잘 일반화됨"

cases = [(0.55, 0.52), (0.99, 0.68), (0.91, 0.88)]
for tr, te in cases:
    print(f"훈련 {tr:.0%} / 테스트 {te:.0%} → {diagnose(tr, te)}")

# 항상 두 점수를 나란히 — 격차가 진단의 열쇠

훈련 55% / 테스트 52% → 과소적합 — 훈련조차 못 맞힘 (더 배워야)
훈련 99% / 테스트 68% → 과적합 — 외웠음! (격차가 큼)
훈련 91% / 테스트 88% → 양호 — 잘 일반화됨


In [8]:
# 트레이드오프 감각 잡기 — 복잡도에 따른 가상의 오차 곡선
def bias_sq(c):     # 복잡도 c가 클수록 편향²은 감소
    return 9.0 / c
def variance(c):   # 복잡도 c가 클수록 분산은 증가
    return 0.5 * c

print("복잡도 | 편향²  분산  합계")
best = None
for c in [1, 2, 3, 4, 6, 9]:
    b, v = bias_sq(c), variance(c)
    total = b + v
    print(f"  {c:2d}   | {b:4.1f}  {v:4.1f}  {total:4.1f}")
    if best is None or total < best[1]:
        best = (c, total)

print(f"→ 균형점: 복잡도 {best[0]} (오차 합 최소)")
# 너무 단순해도(편향↑) 너무 복잡해도(분산↑) 손해 — 중간이 답

복잡도 | 편향²  분산  합계
   1   |  9.0   0.5   9.5
   2   |  4.5   1.0   5.5
   3   |  3.0   1.5   4.5
   4   |  2.2   2.0   4.2
   6   |  1.5   3.0   4.5
   9   |  1.0   4.5   5.5
→ 균형점: 복잡도 4 (오차 합 최소)


# 2026.11.05
인공지능/딥러닝 복습

- 작성자 : 최현준
- 내용 : linear regression을 알아보자

In [1]:
# 공부 시간(x) → 점수(y) 예측
def predict(x, w, b):
    return w * x + b       # y_hat = wx + b

# 학습으로 찾았다고 가정한 값 (기울기 12, 절편 20)
w, b = 12, 20

# 여러 공부 시간에 대해 점수 예측
for hours in [2, 3, 4, 5]:
    score = predict(hours, w, b)
    print(f"{hours}시간 공부 → {score}점 예측")

# 2시간→44점, 3시간→56점... 1시간당 12점씩 (기울기 w)

2시간 공부 → 44점 예측
3시간 공부 → 56점 예측
4시간 공부 → 68점 예측
5시간 공부 → 80점 예측


In [7]:
# 좋은 선인지 숫자로 재기 — 잔차와 MSE
xs = [2, 3, 4, 5]          # 공부 시간
ys = [42, 58, 65, 82]      # 실제 시험 점수 (정답)
w, b = 10, 20


total = 0
for x, y in zip(xs, ys):
    y_hat = w * x + b          # 예측값
    r = y - y_hat              # 잔차 = 실제 - 예측
    total += r ** 2            # 잔차의 제곱을 누적
    print(f"{x}시간: 실제 {y} vs 예측 {y_hat} → 잔차 {r:+d}")

mse = total / len(xs)          # 평균제곱오차
print(f"MSE = {mse}")
# 잔차: -2, +2, -3, +2 → 제곱합 4+4+9+4 = 21 → MSE = 21/4 = 5.25

2시간: 실제 42 vs 예측 40 → 잔차 +2
3시간: 실제 58 vs 예측 50 → 잔차 +8
4시간: 실제 65 vs 예측 60 → 잔차 +5
5시간: 실제 82 vs 예측 70 → 잔차 +12
MSE = 59.25


In [8]:
import math

def sigmoid(z):
    return 1 / (1 + math.exp(-z))   # 1 / (1 + e^-z)

# 여러 입력값을 확률로 변환
for z in [-3, -1, 0, 1, 3]:
    p = sigmoid(z)
    verdict = "범주 A" if p > 0.5 else "범주 B"
    print(f"z={z:2d} → 확률 {p:.2f} → {verdict}")

# z=-3→0.05, z=-1→0.27, z=0→0.50, z=1→0.73, z=3→0.95
# z=0이면 정확히 0.5 — 크면 1에 가깝(A), 작으면 0에 가깝(B)

z=-3 → 확률 0.05 → 범주 B
z=-1 → 확률 0.27 → 범주 B
z= 0 → 확률 0.50 → 범주 B
z= 1 → 확률 0.73 → 범주 A
z= 3 → 확률 0.95 → 범주 A


In [9]:
# 특징 → 점수 → 확률 → 판정 : 분류의 전체 파이프라인
import math

def sigmoid(z):
    return 1 / (1 + math.exp(-z))

# 학습으로 찾았다고 가정한 값 (11장처럼 손으로 지정)
w1, w2, b = 1.2, 0.8, -3.0   # 광고 단어 수 w1, 느낌표 수 w2

mails = [("메일A", 3, 2), ("메일B", 1, 0), ("메일C", 2, 1)]

for name, x1, x2 in mails:
    z = w1 * x1 + w2 * x2 + b            # ① 점수 (wx+b)
    p = sigmoid(z)                        # ② 확률 (0~1)
    verdict = "스팸" if p > 0.5 else "정상"   # ③ 판정 (임계값 0.5)
    print(f"{name}: 점수 {z:+.1f} → 확률 {p:.3f} → {verdict}")

# 메일A: z=+2.2 → 0.900 스팸 / 메일B: z=-1.8 → 0.142 정상
# 메일C: z=+0.2 → 0.550 스팸(아슬아슬) — 임계값을 0.7로 올리면 '정상' 판정

메일A: 점수 +2.2 → 확률 0.900 → 스팸
메일B: 점수 -1.8 → 확률 0.142 → 정상
메일C: 점수 +0.2 → 확률 0.550 → 스팸


# 2026.11.06
인공지능/딥러닝 복습

- 작성자 : 최현준
- 내용 : 경사하강법을 알아보자(제일 막히는 부분)

In [1]:
# 손실 L(w) = (w - 3)^2 를 최소화 (정답은 w=3)
def gradient(w):
    return 2 * (w - 3)      # L의 기울기 (dL/dw)

w = 0.0            # 시작 위치
eta = 0.1          # 학습률(걸음 크기)

for step in range(15):
    w = w - eta * gradient(w)   # w ← w - η·기울기
    print(f"step {step+1:2d}: w = {w:.3f}")

# w가 점점 3에 가까워진다 — 골짜기에 도착!

step  1: w = 0.600
step  2: w = 1.080
step  3: w = 1.464
step  4: w = 1.771
step  5: w = 2.017
step  6: w = 2.214
step  7: w = 2.371
step  8: w = 2.497
step  9: w = 2.597
step 10: w = 2.678
step 11: w = 2.742
step 12: w = 2.794
step 13: w = 2.835
step 14: w = 2.868
step 15: w = 2.894


In [9]:
# 손실 L(w) = (w - 3)^2 를 최소화 (정답은 w=3)
def gradient(w):
    return 2 * (w - 3)      # L의 기울기 (dL/dw)

w = 0.0            # 시작 위치
eta = 0.9          # 학습률(걸음 크기)

for step in range(100):
    w = w - eta * gradient(w)   # w ← w - η·기울기
    print(f"step {step+1:2d}: w = {w:.3f}")

# w가 점점 3에 가까워진다 — 골짜기에 도착!

step  1: w = 5.400
step  2: w = 1.080
step  3: w = 4.536
step  4: w = 1.771
step  5: w = 3.983
step  6: w = 2.214
step  7: w = 3.629
step  8: w = 2.497
step  9: w = 3.403
step 10: w = 2.678
step 11: w = 3.258
step 12: w = 2.794
step 13: w = 3.165
step 14: w = 2.868
step 15: w = 3.106
step 16: w = 2.916
step 17: w = 3.068
step 18: w = 2.946
step 19: w = 3.043
step 20: w = 2.965
step 21: w = 3.028
step 22: w = 2.978
step 23: w = 3.018
step 24: w = 2.986
step 25: w = 3.011
step 26: w = 2.991
step 27: w = 3.007
step 28: w = 2.994
step 29: w = 3.005
step 30: w = 2.996
step 31: w = 3.003
step 32: w = 2.998
step 33: w = 3.002
step 34: w = 2.998
step 35: w = 3.001
step 36: w = 2.999
step 37: w = 3.001
step 38: w = 2.999
step 39: w = 3.000
step 40: w = 3.000
step 41: w = 3.000
step 42: w = 3.000
step 43: w = 3.000
step 44: w = 3.000
step 45: w = 3.000
step 46: w = 3.000
step 47: w = 3.000
step 48: w = 3.000
step 49: w = 3.000
step 50: w = 3.000
step 51: w = 3.000
step 52: w = 3.000
step 53: w =

In [10]:
# 학습률별 궤적 비교 — 같은 산(L=(w-3)^2), 같은 출발점(w=0), 다른 보폭
def step(w, eta):
    return w - eta * 2 * (w - 3)   # w ← w - η·2(w-3)

for eta in [0.01, 0.1, 0.9, 1.1]:
    w = 0.0
    traj = []
    for _ in range(5):
        w = step(w, eta)
        traj.append(f"{w:7.3f}")
    print(f"η={eta:4} :", " ".join(traj))

# 0.01은 굼벵이(5걸음에 0.288), 0.1은 3으로 접근(2.017),
# 0.9는 3을 사이에 두고 진동(5.4→1.08→4.536→…), 1.1은 발산(…→10.465)

η=0.01 :   0.060   0.119   0.176   0.233   0.288
η= 0.1 :   0.600   1.080   1.464   1.771   2.017
η= 0.9 :   5.400   1.080   4.536   1.771   3.983
η= 1.1 :   6.600  -1.320   8.184  -3.221  10.465
